In [ ]:
from torch.optim import SGD, Adam
from torch.optim.lr_scheduler import MultiStepLR
import torch.nn.functional as F
from torch import nn
import torch
import os
import torchvision.transforms as transforms
import math
import pickle
import pandas as pd
import numpy as np 
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
data_1 = pd.read_excel('file_path')

import torch 
print(torch.cuda.is_available())
print(torch.backends.cudnn.version())
print("CUDA version:", torch.version.cuda)
if torch.cuda.is_available():
    print("Current CUDA device:", torch.cuda.get_device_name(torch.cuda.current_device()))

import pandas as pd

# 设置随机种子以确保结果的可复现性
random_state = 42
test_data = data_1.sample(frac=0.3, random_state=random_state)
# 剩余的数据即为训练集
train_data = data_1.drop(test_data.index)
# 打印结果，确保数据被正确划分
print(f"Training set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")

x_train=train_data.iloc[:,18:27].values
z_train=train_data.iloc[:,28:53].values
label_train=train_data.iloc[:,53:56].values
x_test=test_data.iloc[:,18:27].values
z_test=test_data.iloc[:,28:53].values
label_test=test_data.iloc[:,53:56].values
x_train=torch.tensor(x_train, dtype=torch.float)
z_train=torch.tensor(z_train, dtype=torch.float)
label_train=torch.tensor(label_train, dtype=torch.float)
x_test=torch.tensor(x_test, dtype=torch.float)
z_test=torch.tensor(z_test, dtype=torch.float)
label_test=torch.tensor(label_test, dtype=torch.float)
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, x_data, z_data, labels):
        self.x_data = x_data
        self.z_data = z_data
        self.labels = labels

    def __getitem__(self, index):
        return self.x_data[index], self.z_data[index], self.labels[index]

    def __len__(self):
        return len(self.x_data)
    
from torch.utils.data import DataLoader
dataset_train = CustomDataset(x_train, z_train, label_train)
data_loader1 = DataLoader(dataset_train, batch_size=len(label_train), shuffle=True)
dataset_test = CustomDataset(x_test, z_test, label_test)
data_loader2 = DataLoader(dataset_test, len(label_test), shuffle=True)

In [ ]:
from torch.autograd import grad
def cal_loss(model, x, z, y, GR, base, lambd, criterion):
    '''
    仅对ChoiceModel中的PN1施加梯度约束
    
    参数:
    - model: ChoiceModel实例
    - x: 模型输入x（用于FeatureProduce生成x1/x2/x3）
    - z: 模型输入z（用于WeightNN）
    - y: 标签
    - GR: 梯度正则类型（'UGR'/'PGR'/'LGR'/'none'）
    - base: 正则项计算方式（'sum'或其他，对应正梯度和或平方和）
    - lambd: 正则项权重
    - criterion: 基础损失函数（如交叉熵）
    '''
    # 计算基础损失（基于模型最终输出）
    
    choice_probabilities = model(x, z)  
    log_probs = torch.log(choice_probabilities)  
    loss = criterion(log_probs, y)
    
    # 若无需梯度约束，直接返回基础损失
    if GR == None :
        return loss
    
    # 2. 提取PN1的输入和输出（用于计算梯度）
    x1, x2, x3 = model.feature_produce(x)
    x1[torch.isnan(x1)] = 0
    x2[torch.isnan(x2)] = 0
    x3[torch.isnan(x3)] = 0

    # 记录每个PN1的输出（共12个）
    pn1_outputs = []
    # PN1[0-3]对应x1的0-3列
    for i in range(4):
        pn1_out = model.PN1[i](x1[:, i])  # x1[:,i]是PN1[i]的输入
        pn1_outputs.append(pn1_out)
    # PN1[4-8]对应x2的0-4列
    for i in range(5):
        pn1_out = model.PN1[4+i](x2[:, i])  # x2[:,i]是PN1[4+i]的输入
        pn1_outputs.append(pn1_out)
    # PN1[9-11]对应x3的0-2列
    for i in range(3):
        pn1_out = model.PN1[9+i](x3[:, i])  # x3[:,i]是PN1[9+i]的输入
        pn1_outputs.append(pn1_out)
    
    # 根据GR类型计算梯度正则项

    pn1_inputs = []
    pn1_inputs.extend([x1[:, i] for i in range(4)])  # PN1[0-3]的输入
    pn1_inputs.extend([x2[:, i] for i in range(5)])  # PN1[4-8]的输入
    pn1_inputs.extend([x3[:, i] for i in range(3)])  # PN1[9-11]的输入
    
    # 计算每个PN1输出对其输入的梯度
    grads = []
    for idx in range(12):  # 遍历12个PN1
        out = pn1_outputs[idx]  # 第idx个PN1的输出
        inp = pn1_inputs[idx]   # 第idx个PN1的输入（x1/x2/x3的某一列）

        g = grad(
            out, inp, 
            grad_outputs=torch.ones_like(out),  # 梯度权重为1
            create_graph=True, 
            retain_graph=True,
            allow_unused=True  # 允许未使用的输入，避免报错
        )[0]  # 返回梯度张量（形状与inp一致）
        if g is None:
            # 生成与输入形状相同的零张量，且在同一设备上
            g = torch.zeros_like(inp, device=inp.device)
        grads.append(g)  # 此时grads中全是张量，无None

    for idx, (out, inp) in enumerate(zip(pn1_outputs, pn1_inputs)):

        if not inp.grad_fn:
            print(f"警告：PN1[{idx}]的输入特征未参与计算（可能是代码逻辑错误）")
            

    if base == 'sum':
        # 仅累加正梯度（例如约束PN1输出随输入增大而单调递增）
        reg = sum(g[g <= 0].sum() for g in grads)
    else:
        # L2正则（惩罚过大梯度，保证平滑性）
        reg = sum(torch.pow(g, 2).sum() for g in grads)
    
    # 5. 总损失 = 基础损失 + PN1的梯度正则项
    return loss + lambd * reg

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import itertools
import torch.nn.functional as F

# 定义网络架构
class PN1(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, final_hidden_size=64, output_size=1):
        super(PN1, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)  
        self.softplus1 = nn.Softplus()  
        
        self.fc2 = nn.Linear(hidden_size, final_hidden_size)
        self.softplus2 = nn.Softplus()  
        
        self.fc3 = nn.Linear(final_hidden_size, output_size)
    
    def forward(self, x):
        x = x.unsqueeze(1)  
        out = self.fc1(x)
        out = self.softplus1(out) 
        
        out = self.fc2(out)
        out = self.softplus2(out)  
        
        out = self.fc3(out)
        return out
        
class WeightNN(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, output_size0=5, output_size1=6, output_size2=4):
        super(WeightNN, self).__init__()
        self.hidden1 = nn.Linear(input_size, hidden_size1)
        self.hidden2_para1 = nn.Linear(hidden_size1, hidden_size2)
        self.hidden2_para2 = nn.Linear(hidden_size1, hidden_size2)
        self.hidden2_para3 = nn.Linear(hidden_size1, hidden_size2)
        self.output0 = nn.Linear(hidden_size2, output_size0)
        self.output1 = nn.Linear(hidden_size2, output_size1)
        self.output2 = nn.Linear(hidden_size2, output_size2)
        self.output3 = nn.Linear(hidden_size1, 3)

    def forward(self, z):
        N = len(z)
        f2 = torch.zeros(N, 12, device=z.device)
        f4 = torch.zeros(N, 3, device=z.device)
        f = F.leaky_relu(self.hidden1(z))
        f1_0 = F.leaky_relu(self.hidden2_para1(f))
        f2_0 = torch.sigmoid(self.output0(f1_0))
        f1_1 = F.leaky_relu(self.hidden2_para2(f))
        f2_1 = torch.sigmoid(self.output1(f1_1))
        f1_2 = F.leaky_relu(self.hidden2_para3(f))
        f2_2 = torch.sigmoid(self.output2(f1_2))
        f3 = self.output3(f)

        f2[:, 0] = -f2_0[:, 0]
        f2[:, 1] = -f2_0[:, 1]
        f2[:, 2] = f2_0[:, 2]
        f2[:, 3] = f2_0[:, 3]
        f2[:, 4] = -f2_1[:, 0]
        f2[:, 5] = -f2_1[:, 1]
        f2[:, 6] = f2_1[:, 2]
        f2[:, 7] = f2_1[:, 3]
        f2[:, 8] = f2_1[:, 4]
        f2[:, 9] = -f2_2[:, 0]
        f2[:, 10] = -f2_2[:, 1]
        f2[:, 11] = f2_2[:, 2]
        
        f4[:,0] = f2_0[:, 4]
        f4[:,1] = f2_1[:, 5]
        f4[:,2] = f2_2[:, 3]
        
        

        return f2, f4, f3

class FeatureProduce(nn.Module):
    def __init__(self):
        super(FeatureProduce, self).__init__()

    def forward(self, x):


        x1 = torch.cat([
            x[:, 0:3],  # 前3列（直接来自x，保留梯度）
            (x[:, 1] / x[:, 0]).unsqueeze(1)  
        ], dim=1)
        

        x2 = torch.cat([
            x[:, 3:7],  # 第3-6列
            (x[:, 4] / x[:, 3]).unsqueeze(1)  # 第4列
        ], dim=1)
        

        x3 = torch.cat([
            x[:, 7:9],  # 第7-8列
            (x[:, 8] / x[:, 7]).unsqueeze(1)  # 第2列
        ], dim=1)
        
        x1 = torch.where(torch.isnan(x1), torch.zeros_like(x1), x1)
        x2 = torch.where(torch.isnan(x2), torch.zeros_like(x2), x2)
        x3 = torch.where(torch.isnan(x3), torch.zeros_like(x3), x3)
        
        return x1, x2, x3
    
class ChoiceModel(nn.Module):
    def __init__(self):
        super(ChoiceModel, self).__init__()
        self.WeightNN = WeightNN(input_size=25, hidden_size1=100, hidden_size2=100, output_size0=5, output_size1=6, output_size2=4)
        self.PN1 = nn.ModuleList([PN1() for _ in range(12)])
        self.feature_produce = FeatureProduce()


    def forward(self, x, z):
        weight1, weight2, bias = self.WeightNN(z)
        x1, x2, x3 = self.feature_produce(x)

        
        x1[torch.isnan(x1)] = 0
        x2[torch.isnan(x2)] = 0
        x3[torch.isnan(x3)] = 0
        
        w1 = weight1[:, 0:4]
        w2 = weight1[:, 4:9]
        w3 = weight1[:, 9:12]

        ww1= weight2[:,0].unsqueeze(1)
        ww2= weight2[:,1].unsqueeze(1)
        ww3= weight2[:,2].unsqueeze(1)
        b1 = bias[:, 0].unsqueeze(1)
        b2 = bias[:, 1].unsqueeze(1)
        b3 = bias[:, 2].unsqueeze(1)

        PP1=self.PN1[1](x1[:, 1])
        w11=w1[:,1].unsqueeze(1)
        temp=PP1*w11
        
        F1 = sum(w1[:, i].unsqueeze(1) * self.PN1[i](x1[:, i]) for i in range(4)) +b1
        F2 = sum(w2[:, i].unsqueeze(1) * self.PN1[4 + i](x2[:, i]) for i in range(5)) + b2
        F3 = sum(w3[:, i].unsqueeze(1) * self.PN1[9 + i](x3[:, i]) for i in range(3)) + b3

        utilities = torch.cat([F1, F2, F3], dim=1)
        choice_probabilities = F.softmax(utilities,dim=1)
        return choice_probabilities
        
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ChoiceModel()
model.to(device)

In [ ]:
x_train = x_train.to(device)
z_train = z_train.to(device)
label_train = label_train.to(device)
print("Model device:", next(model.parameters()).device)
print("x_train device:", x_train.device)
print("z_train device:", z_train.device)
print("label_train device:", label_train.device)
print(model(x_train, z_train))
print(np.shape(model(x_train, z_train)))

In [ ]:
from torchmetrics.functional.classification import multiclass_f1_score
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
criterion = nn.NLLLoss(reduction='mean')  # 基础损失：负对数似然
n_epochs = 1
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = MultiStepLR(optimizer, milestones=[0.5 * n_epochs, 0.75 * n_epochs], gamma=0.1)



GR = 'UGR'
base = 'sum'  # 正则项计算方式（正梯度和）
lambd = 0.1  # 正则项权重（控制约束强度）


def evaluate(model, data_loader, criterion, device):
    """计算数据集的准确率、F1分数、平均NLL损失"""
    model.eval()
    correct = 0
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for x, z, y in data_loader:
            x = x.to(device)
            z = z.to(device)
            y = y.to(device)
            
            output = model(x, z)
            pred = output.argmax(-1)
            y = torch.argmax(y, dim=1)  # one-hot转类别索引
            
            correct += pred.eq(y).sum().item()
            log_probs = torch.log(output)
            total_loss += criterion(log_probs, y).item() * x.size(0)
            
            all_preds.append(pred.cpu())
            all_labels.append(y.cpu())
    
    total = len(data_loader.dataset)
    acc = correct / total
    avg_loss = total_loss / total
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    f1 = multiclass_f1_score(all_labels, all_preds, num_classes=3)
    
    return acc, f1, avg_loss



def train(model, train_loader, optimizer, criterion, device, GR, base, lambd, n_epochs):
    model.train()
    Loss_=[]
    for epoch in range(n_epochs):
        for x, z, y in train_loader:
            x = x.to(device)
            z = z.to(device)
            y = y.to(device)
            y = torch.argmax(y, dim=1)  # 处理标签
            x.requires_grad_(True)  # 新增此行，开启x的梯度跟踪
            
            optimizer.zero_grad()
            loss = cal_loss(model, x, z, y, GR, base, lambd, criterion)
            loss.backward()
            optimizer.step()
        Loss_.append(loss.detach().cpu().item())
        scheduler.step()  # 每轮结束后更新学习率
    print(f"训练完成（共 {n_epochs} 轮），NLL 为{loss}")
    return Loss_

In [ ]:

# 开始训练
train(model, data_loader1, optimizer, criterion, device, GR, base, lambd, n_epochs=2000)

# 训练结束后，分别评估训练集和测试集
train_acc, train_f1, train_nll = evaluate(model, data_loader1, criterion, device)
test_acc, test_f1, test_nll = evaluate(model, data_loader2, criterion, device)

# 统一输出结果
print("\n" + "="*50)
print("最终指标对比")
print("-"*50)
print(f"训练集：")
print(f"  准确率：{train_acc:.4f} ({100*train_acc:.1f}%)")
print(f"  F1分数：{train_f1:.4f}")
print(f"  NLL损失：{train_nll:.4f}")
print("-"*50)
print(f"测试集：")
print(f"  准确率：{test_acc:.4f} ({100*test_acc:.1f}%)")
print(f"  F1分数：{test_f1:.4f}")
print(f"  NLL损失：{test_nll:.4f}")
print("="*50)

In [ ]:
total_loss =[]
train_time = 50
for i in range(train_time):
    # 训练模型，会更新全局变量Loss_（张量类型）
    model = ChoiceModel()
    model.to(device)
    criterion = nn.NLLLoss(reduction='mean')  # 基础损失：负对数似然
    n_epochs = 1
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = MultiStepLR(optimizer, milestones=[0.5 * n_epochs, 0.75 * n_epochs], gamma=0.1)
    Loss_= train(model, data_loader1, optimizer, criterion, device, GR, base, lambd, n_epochs=2000)
    total_loss.append( Loss_)

In [ ]:
file_path = 'save_path'   
torch.save(model.state_dict(), file_path)

temp = torch.zeros(XX.shape[0], XX.shape[1])
for i in range(12):
    temp[:,i]=model.PN1[i](XX[:,i]).squeeze(1)  

np1_gc_result = temp.detach().cpu().numpy()
file_path2 = 'result_path'   
np.savetxt(file_path2, np1_gc_result, delimiter=',')

np1_gc_input=XX.detach().cpu().numpy()
file_path3 = 'input_path'   
np.savetxt(file_path3, np1_gc_input, delimiter=',')

In [ ]:
import pandas as pd
import pysr
import sympy
import numpy as np
from matplotlib import pyplot as plt
from pysr import PySRRegressor
from sklearn.model_selection import train_test_split

Name=['TRAIN TT','TRAIN CO','TRAIN HE','TRAIN VOT','SM TT','SM CO','SM HE','SM SEATS','SM VOT','CAR TT','CAR CO','CAR VOT']
Input_1 = pd.read_csv('file_path2')
output_1 = pd.read_csv('file_path3')
# Find the latent function to approximate
X=Input_1['TRAIN TT'].values
Y=output_1['TRAIN TT'].values
X=torch.tensor(X).unsqueeze(1)
Y=torch.tensor(Y).unsqueeze(1)

In [ ]:
import julia
import sympy as sp
from pysr import PySRRegressor
psmodel = PySRRegressor(
    # 1. 核心搜索参数
    niterations=50,
    population_size=50,
    tournament_selection_n=2,
    
    # 2. 扩展函数基
    unary_operators=[
        "square",    # x²
        "cube",      # x³
        "exp",       # e^x
        "log",       # log(x)（配合abs避免负数）
        "abs",       # |x|（分段线性，增加灵活性）
        "inv",  
        "identity",  # 新增：帮助学习线性项
    ],
    binary_operators=[
        "+",         # x + y
        "*",         # x * y
        "-",         # x - y
        # 注意：这里不再包含 "iflt_3"
    ],
    # 3. 使用内置 ifelse
    extra_sympy_mappings={
        "ifelse": lambda cond, a, b: sp.Piecewise((a, cond), (b, True))
    },
    
    # 4. 约束复杂度
    maxsize=12,
    maxdepth=8,
    
    # 5. 实用配置
    random_state=42,
    progress=True,
)

In [ ]:
psmodel.fit(X, Y)
best_equation = model.get_best()
print(best_equation)
best_equation.equation

# When the latent function was approximated with SR, replace the PN1[i] with learned function and train model again